# PHD — Intrinsic Dimension Estimation for Robust Detection of AI-Generated Texts

Tulchinskii et al., **NeurIPS 2023** ·
[proceedings.neurips.cc](https://proceedings.neurips.cc/paper_files/paper/2023/hash/7baa48bc166aa2013d78cbdc15010530-Abstract-Conference.html) ·
arXiv: https://arxiv.org/abs/2306.04723

> **Phạm vi**: chỉ inference + eval theo bộ metric của supervised. Chưa eval dưới attack.
> Nhánh **Thống kê/Zero-Shot**, không train — chỉ dùng 1 encoder có sẵn (frozen) để lấy embedding.

## Cơ chế (Persistent Homology Dimension)

Coi tập embedding của các token trong 1 văn bản là 1 đám mây điểm trên manifold — ước lượng
**intrinsic dimension** của manifold đó bằng tốc độ tăng của tổng trọng số **cây khung nhỏ nhất**
(Minimum Spanning Tree) khi tăng dần kích thước mẫu con:

```
E(m) ~ m^{(d-1)/d}          với m điểm lấy mẫu con, d = intrinsic dimension
=> log E(m) = ((d-1)/d) log m + const   -> hồi quy tuyến tính log-log lấy slope -> suy ra d
```

Phát hiện chính của paper: **văn bản người có PHD trung bình ~9** (Latin-script), **AI thấp hơn ~1.5**.
Notebook này dùng `score = -PHD(x)` (PHD thấp = AI → đảo dấu để "cao = AI" khớp label=1).

## Deviation so với paper

| # | Paper | Ta dùng | Lý do |
|---|---|---|---|
| 1 | Encoder RoBERTa-base (paper dùng đúng model này) | **`roberta-base`** | Khớp paper, không đổi |
| 2 | Layer/embedding cụ thể dùng làm point cloud không nêu chi tiết trong bản tóm tắt ta có | **`hidden_states[-1]`** (layer cuối) | Xấp xỉ hợp lý — layer cuối mang thông tin ngữ nghĩa đủ giàu; ghi rõ đây là lựa chọn xấp xỉ, chưa đối chiếu trực tiếp với code gốc |
| 3 | Số lần lặp lại mỗi kích cỡ mẫu con (`num_repeats`) không nêu rõ | **`NUM_REPEATS = 3`** | Hạ tầng — đủ để làm mượt nhiễu ước lượng MST mà không tốn quá nhiều compute/mẫu |

Phần MST/hồi quy log-log chạy trên **CPU** (scipy) — dual-GPU sharding vẫn giúp vì nó chia đôi cả
phần forward-pass lấy embedding lẫn số dòng mỗi process phải chạy MST tuần tự.

⚡ **Dual T4 (2 GPU)**: notebook này chấm điểm theo lối **sharding** — 2 process độc lập, mỗi process
1 GPU, mỗi process chỉ xử lý ~1/2 số dòng (không cần đồng bộ gradient/NCCL vì mỗi dòng độc lập).
Chạy qua `torchrun --nproc_per_node=2`, giống cơ chế launch của `supervised_models/` nhưng không có
bước `dist.init_process_group`/all-reduce nào — chỉ dùng `RANK`/`WORLD_SIZE` để chia việc.


In [ ]:
!pip install -q -U "transformers>=4.45" accelerate sentencepiece scipy

In [ ]:
%%writefile score_worker.py
import os, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

# ── Bản dataset dùng để train/eval ────────────────────────────────────
#   "deconf2"    = 02c_deconfound_v2.ipynb  <- ĐANG DÙNG
#                  canonicalize + truncate_pair_sentences + lọc notation_gap 85%.
#                  shortcut_audit.py đo đủ 22 đặc trưng: tầng process 0.5055 (cổng
#                  cứng, PASS) · tổng 0.8909. Train 7.025 cặp (85% của 8.265).
#   "deconf"     = 02b_deconfound.ipynb. ĐỪNG dùng cho nhánh này. Hai lý do:
#                  (a) con số "surface-GBM 0.6867, G6 PASS" mà 02b tự báo chỉ đo 6
#                      đặc trưng, thiếu avg_word_len (0.7795) lẫn type_token_ratio
#                      (0.7727). Đo đủ 22 thì là 0.9114 -> WARN, không PASS.
#                  (b) truncate_pair cắt thô đúng từ thứ min(len) nên chỉ 52,4% văn
#                      bản còn kết thúc đúng dấu câu (trước khi cắt: 99,0%). Detector
#                      ở file này chấm điểm bằng log-probability của từng token, nên
#                      token bị chặt giữa câu là perplexity GIẢ — và nó BẤT ĐỐI XỨNG,
#                      vì câu AI dài hơn (22,4 vs 20,4 từ) nên bên AI mất nhiều từ
#                      hơn một cách có hệ thống.
#   "normalized" = bản GỐC, CÒN shortcut: chỉ ĐẾM SỐ TỪ đã đoán đúng human/AI ở
#                  AUROC 0.7193 (surface-GBM 0.8080). G6 FAIL.
#
# PROVENANCE của số cũ (REPORT_2026-08-01.md §5) — đã truy được, ĐỪNG suy diễn lại:
# log còn lưu trong output notebook ghi sha256 val=c9d3957ae20b ood=2e9a47ca22e0
# fp=584cb43da181, khớp ĐÚNG dataset_{val,test_ood_model,test_fp}_normalized.parquet.
# Tức mọi số static/detectgpt trong báo cáo đó chạy trên "normalized" — chưa
# canonicalize, chưa cắt cặp — KHÔNG phải "deconf" như dòng tag cũ ghi (dòng đó nằm
# trên %%writefile nên chưa bao giờ tới được worker).
# Hệ quả: AUROC(val) < 0.5 của Fast-DetectGPT (0.4760) và Binoculars (0.4650) KHÔNG
# do tiền xử lý. Nguyên nhân phải tìm ở chỗ khác — mismatch thời đại backbone (gpt2-xl
# 2019 / gpt-j 2021 / falcon 2023 vs generator 2026), hoặc bug implementation. PHẢI có
# positive control trên một tập đã biết kết quả công bố trước khi tin bảng so sánh.
#
# PHẢI trùng tag với supervised_models/ và adversarial_models/ — cả 8 notebook đó đã
# chạy trên "deconf2". Lệch tag thì hai bảng số KHÔNG so được với nhau: deconf 8.265
# cặp vs deconf2 7.025 cặp, và nội dung text cũng khác (cắt theo từ vs cắt biên câu).
#
# PHẢI đặt TRONG body của %%writefile, KHÔNG phải ở đầu cell notebook. Đặt ở đầu cell
# thì %%writefile bị đẩy khỏi dòng 0 -> Jupyter hiểu thành line magic và báo "Line
# magic function `%%writefile` not found"; mà kể cả chạy được thì worker vẫn NameError
# vì nó là tiến trình riêng, không thấy biến của notebook.
#
# LƯU Ý: Kaggle dataset phải chứa dataset_{train,val,test_ood_model,test_fp}_deconf2.parquet
DATASET_TAG = "deconf2"
ENCODER_MODEL = "roberta-base"
MAX_LEN = 512
MIN_SUBSAMPLE = 40
STEP = 10
NUM_REPEATS = 3
RUN_NAME = "phd"
MODEL_DESC = ENCODER_MODEL

RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
DEVICE = f"cuda:{LOCAL_RANK}" if torch.cuda.is_available() else "cpu"
SMOKE_TEST = os.environ.get("SMOKE", "0") == "1"
SMOKE_PER_CLASS = 16
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def log(*a):
    print(f"[rank{RANK}]", *a, flush=True)

from transformers import AutoTokenizer, AutoModel
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import cdist

tok = AutoTokenizer.from_pretrained(ENCODER_MODEL)
encoder = AutoModel.from_pretrained(ENCODER_MODEL).to(DEVICE).eval()
log("loaded:", ENCODER_MODEL, "on", DEVICE)

def _mst_length(points):
    d = cdist(points, points)
    return minimum_spanning_tree(d).sum()

def _phd_estimate(points, min_points=MIN_SUBSAMPLE, num_repeats=NUM_REPEATS, step=STEP, rng=None):
    n = points.shape[0]
    if n <= min_points + step:
        return np.nan
    rng = rng or np.random.default_rng(0)
    sizes = list(range(min_points, n, step))
    if sizes[-1] != n:
        sizes.append(n)
    log_n, log_e = [], []
    for m in sizes:
        lengths = [_mst_length(points[rng.choice(n, size=m, replace=False)]) for _ in range(num_repeats)]
        log_n.append(np.log(m))
        log_e.append(np.log(np.mean(lengths) + 1e-9))
    slope, _ = np.polyfit(log_n, log_e, 1)
    return 1.0 / (1.0 - slope) if slope < 1 else np.nan

@torch.no_grad()
def _embed_tokens(text):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)
    out = encoder(**enc, output_hidden_states=True)
    return out.hidden_states[-1][0].float().cpu().numpy()

def score(texts):
    """score cao = AI. Hướng ĐĂNG KÝ TRƯỚC theo paper (Tulchinskii et al., NeurIPS 2023:
    văn bản người có intrinsic dimension CAO hơn AI) -> score = -PHD. Không suy hướng từ
    nhãn val — xem statistical_branch_plan.md §4.

    _phd_estimate trả NaN khi không ước lượng được: text <= MIN_SUBSAMPLE+STEP token, hoặc
    slope >= 1. Trả NaN, KHÔNG trả 0.0. Lý do: -PHD nằm khoảng -8..-12, nên 0.0 CAO HƠN mọi
    mẫu đo được -> mẫu không đo được bị xếp thành "giống AI nhất" trong ROC. Nếu văn bản
    ngắn lệch về một lớp thì đó là bias có hệ thống. Cell metric loại NaN và báo số lượng.
    """
    out, n_nan = [], 0
    for text in texts:
        pts = _embed_tokens(text)
        d = _phd_estimate(pts)
        if np.isnan(d):
            n_nan += 1
            out.append(np.nan)
        else:
            out.append(-d)
    if n_nan:
        log(f"[!] {n_nan}/{len(texts)} mau khong uoc luong duoc PHD -> score = NaN (bi loai o metric)")
    return np.array(out)

# ── Data + sharding + chấm điểm + lưu partial (mỗi rank ghi 1 file riêng) ────
KAGGLE_INPUT = "/kaggle/input/datasets/<kaggle-user>/halfdatadetector"

def find_data_dir():
    target = f"dataset_val_{DATASET_TAG}.parquet"
    if (Path(KAGGLE_INPUT) / target).exists():
        return Path(KAGGLE_INPUT)
    for base in [Path(KAGGLE_INPUT), Path("output"), Path("."), Path("/kaggle/input")]:
        if base.exists():
            hits = list(base.rglob(target))
            if hits:
                return hits[0].parent
    raise FileNotFoundError(f"Khong tim thay {target}. Kiem tra KAGGLE_INPUT = {KAGGLE_INPUT}")

DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}"
OUT_DIR.mkdir(parents=True, exist_ok=True)   # exist_ok=True -> an toàn dù 2 rank cùng gọi
log("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

def melt_pairs(df):
    h = pd.DataFrame({"text": df["human_text"], "label": 0, "paper_id": df["paper_id"],
                       "ai_degree": 0, "generator_family": "human"})
    a = pd.DataFrame({"text": df["ai_text"], "label": 1, "paper_id": df["paper_id"],
                       "ai_degree": df.get("ai_degree", 3),
                       "generator_family": df.get("generator_family", "unknown")})
    out = pd.concat([h, a], ignore_index=True)
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def melt_fp(df):
    has_ai = df["ai_text"].notna() if "ai_text" in df.columns else pd.Series(False, index=df.index)
    text = df["ai_text"].where(has_ai, df["human_text"]) if "ai_text" in df.columns else df["human_text"]
    out = pd.DataFrame({"text": text, "label": 0, "paper_id": df["paper_id"],
                         "ai_degree": df.get("ai_degree", 0).fillna(0).astype(int) if "ai_degree" in df.columns else 0,
                         "generator_family": "human"})
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def load(name, fp_mode=False):
    p = DATA_DIR / f"dataset_{name}_{DATASET_TAG}.parquet"
    if not p.exists():
        log(f"thieu {p.name} - bo qua"); return None
    df = pd.read_parquet(p)
    return melt_fp(df) if fp_mode else melt_pairs(df)

def maybe_smoke(df):
    if df is None or not SMOKE_TEST:
        return df
    n = min(SMOKE_PER_CLASS, int(df["label"].value_counts().min()))
    return (df.groupby("label", group_keys=False)
              .sample(n=n, random_state=SEED)
              .sample(frac=1, random_state=SEED)
              .reset_index(drop=True))

for split_name, fp_mode in [("val", False), ("test_ood_model", False), ("test_fp", True)]:
    df = load(split_name, fp_mode=fp_mode)
    df = maybe_smoke(df)
    if df is None or len(df) == 0:
        continue
    shard = df.iloc[RANK::WORLD_SIZE].reset_index(drop=True)   # chia dòng, không chia theo model
    log(f"{split_name}: {len(df)} dong tong -> shard {len(shard)} dong (rank {RANK}/{WORLD_SIZE})")
    if len(shard) == 0:
        continue
    shard["score"] = score(shard["text"].tolist())
    if SMOKE_TEST:
        log(f"[SMOKE] KHONG luu partial cho {split_name} ({len(shard)} dong) - chi de kiem tra chay duoc.")
        continue
    out_path = OUT_DIR / f"partial_{split_name}_rank{RANK}.parquet"
    shard[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(out_path, index=False)
    log(f"da luu {out_path.name} ({len(shard)} dong)")

log("XONG phan cua rank nay.")

# ═══ CHẾ ĐỘ MIX — chấm tập trộn của 06_mixing để dựng đường dose-response ═════════
# Bật bằng: RUN_MIX=1 <lệnh chạy worker như thường>
#
# Vì sao nằm ở worker này chứ không ở 07_score_mix: 07 load checkpoint bằng
# AutoModelForSequenceClassification. Bốn detector zero-shot KHÔNG có checkpoint
# (falcon/gpt-j đóng băng), nên chúng phải được chấm bằng chính score() ở trên. Và PHẢI là
# score() đó, không viết lại — hai đường trong cùng một figure mà dùng hai hàm chấm khác
# nhau thì figure không nói được gì.
#
# Tập mix KHÔNG đi qua vòng split thường: text đã được 06 assemble sẵn nên không melt, và
# không có label nhị phân — chỉ có mix_ratio_actual liên tục. Vòng split thường tính
# AUROC/τ, vô nghĩa ở đây. Nên mix ghi riêng pred_mix*.parquet và KHÔNG vào metrics.json.
#
# Schema đầu ra = ĐÚNG 15 cột KEEP của 07_score_mix + score -> gộp chung được với
# scores_all.parquet mà không lệch cột.
MIX_NAME = "dataset_test_ood_model_mix.parquet"
MIX_KEEP = ["sample_id", "paper_id", "source", "generator_family", "mix_structure",
            "mix_seed", "mix_k_slots", "n_slots", "mix_ratio_actual", "n_seams",
            "word_count", "align_score", "was_mixed", "ai_degree", "in_spec_band"]
RUN_MIX = os.environ.get("RUN_MIX", "0") == "1"
# 0 = FULL (20.358 dòng). Đặt số n để lấy n dòng MỖI Ô (bin x mix_structure) = 20 ô.
MIX_PER_CELL = int(os.environ.get("MIX_PER_CELL", "0"))


def find_mix_file():
    """mix.parquet ở old_dataset/ khi chạy local, ở /kaggle/input khi chạy Kaggle."""
    for base in [Path(KAGGLE_INPUT), Path("/kaggle/input"), Path("."), Path(".."),
                 Path("old_dataset"), Path("output")]:
        try:
            if base.exists():
                hits = sorted(base.rglob(MIX_NAME))
                if hits:
                    return hits[0]
        except OSError:
            continue
    return None


def load_mix():
    p = find_mix_file()
    if p is None:
        log(f"khong thay {MIX_NAME} -> bo qua che do mix")
        return None, None
    df = pd.read_parquet(p)
    have = [c for c in MIX_KEEP if c in df.columns]
    missing = [c for c in MIX_KEEP if c not in df.columns]
    if missing:
        log(f"[!] mix thieu cot {missing} -> de NaN de concat voi 07 khong lech cot")
    out = df[["text"] + have].copy()
    for c in missing:
        out[c] = np.nan
    out = out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)]
    if MIX_PER_CELL > 0:
        _b = (out["mix_ratio_actual"].fillna(0.0) * 10).astype(int).clip(0, 9)
        out = (out.assign(_bin=_b)
                  .groupby(["_bin", "mix_structure"], group_keys=False)
                  .apply(lambda g: g.sample(n=min(len(g), MIX_PER_CELL), random_state=SEED))
                  .drop(columns="_bin"))
        log(f"mix: subsample {MIX_PER_CELL} dong/o (bin x structure)")
    return out.reset_index(drop=True), p


def run_mix():
    import math as _math
    import time as _time

    _rank = int(globals().get("RANK", 0))
    _ws = int(globals().get("WORLD_SIZE", 1))
    df, src = load_mix()
    if df is None or len(df) == 0:
        return
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    shard = df.iloc[_rank::_ws].reset_index(drop=True) if _ws > 1 else df
    log(f"mix: {len(df)} dong tong -> {len(shard)} dong cua rank {_rank}/{_ws}  <- {src}")
    if SMOKE_TEST:
        _s = score(shard["text"].tolist()[:8])
        log(f"[SMOKE] mix: 8 dong, finite={np.isfinite(_s).all()}, "
            f"min/max={float(np.min(_s)):.4f}/{float(np.max(_s)):.4f}; KHONG luu.")
        return

    ck = OUT_DIR / f"chunks_mix_rank{_rank}"
    ck.mkdir(parents=True, exist_ok=True)
    step = int(globals().get("CHECKPOINT_ROWS", 0) or 200)
    n_ch = _math.ceil(len(shard) / step)
    log(f"mix: {n_ch} chunk x {step} dong, checkpoint tai {ck}")
    t_all = _time.time()
    for k in range(n_ch):
        a, b = k * step, min((k + 1) * step, len(shard))
        p = ck / f"mix_chunk{k:05d}.parquet"
        if p.exists():
            log(f"resume mix chunk {k + 1}/{n_ch}: da co")
            continue
        part = shard.iloc[a:b].copy()
        t0 = _time.time()
        part["score"] = score(part["text"].tolist())
        tmp = p.with_suffix(".tmp.parquet")
        part.drop(columns=["text"]).to_parquet(tmp, index=False)
        os.replace(tmp, p)          # atomic -> het gio session khong de lai chunk viet do dang
        rate = (b - a) / max(_time.time() - t0, 1e-9)
        log(f"mix chunk {k + 1}/{n_ch}: {rate:.2f} mau/s, "
            f"ETA con lai ~{(len(shard) - b) / max(rate, 1e-9) / 3600:.2f}h")

    parts = sorted(ck.glob("mix_chunk*.parquet"))
    if parts:
        allp = pd.concat([pd.read_parquet(x) for x in parts], ignore_index=True)
        dst = OUT_DIR / (f"pred_mix_rank{_rank}.parquet" if _ws > 1 else "pred_mix.parquet")
        allp.to_parquet(dst, index=False)
        log(f"da luu {dst.name}: {len(allp)} dong, wall={_time.time() - t_all:.0f}s")
        log(f"     cot: {list(allp.columns)}")


if RUN_MIX:
    run_mix()

## Chạy trên dual T4 (Kaggle)

Script đã viết ra `score_worker.py` ở cell trên **tự chia việc theo `RANK`/`WORLD_SIZE`** — mỗi
GPU chỉ chấm điểm ~1/2 số dòng, không cần đồng bộ giữa 2 process (mỗi dòng độc lập, không có
gradient nào phải all-reduce) — khác DDP lúc train ở `supervised_models/`, ở đây chỉ là **chia
việc song song**.

1. **SMOKE** (kiểm 2 GPU bắt tay + không OOM, ~1-2 phút): chạy cell `SMOKE=1 torchrun ...`.
2. **FULL**: chạy cell `torchrun ...` (không `SMOKE=1`) — chấm điểm toàn bộ `val`/`test_ood_model`/`test_fp`.
3. Sau khi cả 2 rank chạy xong (torchrun tự đợi), chạy cell **gộp kết quả** rồi tới cell **metrics**.

⚠️ Nếu máy/Kaggle session chỉ có **1 GPU**, đổi `--nproc_per_node=2` → `--nproc_per_node=1`
(vẫn đúng, chỉ là không còn chạy song song 2 GPU — chậm hơn nhưng không lỗi).


In [ ]:
# ── Checkpoint của lần chạy trước: PHẢI xoá khi đổi DATASET_TAG ───────────────
# Chunk/partial cũ mang _data_sha256 của file parquet CŨ. Notebook nào có
# validate_chunk() thì raise; notebook nào không có thì lẳng lặng gộp score của
# dataset cũ vào metric của dataset mới — đúng loại bug im lặng tệ nhất.
#
# Cell này KHÔNG tự xoá. Nó liệt kê ra để bạn nhìn trước, rồi bạn bật cờ.
import shutil
from pathlib import Path

PURGE_STALE = False   # đặt True MỘT LẦN sau khi đổi tag, chạy cell, rồi trả lại False

_base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
_ckpt, _derived = [], []
for _d in sorted(_base.glob("artifacts_*")):
    _ckpt += [p for p in (_d / "chunks",) if p.is_dir()]
    _ckpt += sorted(_d.glob("partial_*.parquet")) + sorted(_d.glob("chunk_*.parquet"))
    _derived += sorted(_d.glob("pred_*.parquet")) + sorted(_d.glob("metrics*.json"))

if not _ckpt and not _derived:
    print("sạch — không có kết quả của lần chạy trước.")
else:
    print(f"CHECKPOINT (score thô, phải xoá): {len(_ckpt)}")
    for _p in _ckpt:
        print("   ", _p)
    print(f"DẪN XUẤT (metric/pred, bị ghi đè khi chạy lại đủ 3 split): {len(_derived)}")
    for _p in _derived:
        print("   ", _p)
    if PURGE_STALE:
        for _p in _ckpt + _derived:
            shutil.rmtree(_p) if _p.is_dir() else _p.unlink()
        print(f"\nĐÃ XOÁ {len(_ckpt) + len(_derived)} mục.")
    else:
        print("\nPURGE_STALE = False -> chưa xoá gì. Bật True rồi chạy lại cell này.")

In [ ]:
# ── SMOKE: kiểm 2 GPU bắt tay + đọc được parquet, vài phút ──────────────────────
# RUN_MIX=1 ở đây để SMOKE kiểm LUÔN việc tìm thấy dataset_test_ood_model_mix.parquet
# và chấm được 8 dòng của nó. Không có nó thì bạn chỉ phát hiện "quên upload mix" SAU
# khi benchmark đã chạy hàng giờ — worker bỏ qua mix im lặng, không báo lỗi.
# Phải thấy 3 dòng sha256/số dòng của deconf2 VÀ một dòng "[SMOKE] mix: 8 dong".
!RUN_MIX=1 SMOKE=1 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 score_worker.py

In [ ]:
# ── CHẠY FULL: 3 split benchmark + tập MIX trong MỘT lần gọi ────────────────────
# RUN_MIX=1 nằm ở đây thay vì một cell riêng vì worker load model ở module level. Tách
# hai lệnh thì PHD load model HAI lần; gộp lại thì load một lần rồi chấm
# splits -> mix liền. Block mix nằm cuối lastde/score_worker.py nên thứ tự tự đúng.
#
# Ước tính: ~20ph + ~50ph = ~1,2h. Dưới hạn 12h/session của Kaggle.
# Checkpoint chunk + atomic replace ở CẢ HAI phần -> session chết thì chạy lại cell này
# là tiếp tục đúng chỗ, không chấm lại gì.
# Muốn rút tập mix: thêm MIX_PER_CELL=150 (150 dòng mỗi ô bin x structure = 3.000 dòng).
!RUN_MIX=1 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 score_worker.py

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

RUN_NAME = "phd"
MODEL_DESC = "roberta-base"
TARGET_FPR = 0.01

def _merge_partials(split_name, world_size=2):
    parts = []
    for r in range(world_size):
        p = OUT_DIR / f"partial_{split_name}_rank{r}.parquet"
        if p.exists():
            parts.append(pd.read_parquet(p))
    return pd.concat(parts, ignore_index=True) if parts else None

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}"

val_df = _merge_partials("val")
ood_df = _merge_partials("test_ood_model")
fp_df = _merge_partials("test_fp")
for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if d is not None:
        print(f"{nm:<16} {len(d):>7,} mau (gop tu 2 rank)  label={d['label'].value_counts().to_dict()}")


In [ ]:
# ── Metrics: CÙNG công thức với supervised_models (tau@FPR=1% trên val) ──────
# score() đã được tính sẵn trong score_worker.py (dual-GPU) và gộp lại ở cell trên.
def tpr_at_fpr(y, s, fpr):
    y, s = np.asarray(y), np.asarray(s); neg = s[y == 0]
    if len(neg) == 0 or (y == 1).sum() == 0:
        return np.nan, np.nan
    thr = np.quantile(neg, 1 - fpr, method="higher")
    return float((s[y == 1] >= thr).mean()), float(thr)

def auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan

def report_split(name, d):
    if d is None:
        return None
    y, s = d["label"].values, d["score"].values
    row = {"split": name, "n": len(d), "n_ai": int((y == 1).sum()),
           "auroc": auroc(y, s),
           "auprc": float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else np.nan}
    for f in [0.001, 0.01, 0.05]:
        row[f"tpr@fpr{f}"] = tpr_at_fpr(y, s, f)[0]
    row["tpr@tau"] = float((s[y == 1] >= TAU).mean()) if (y == 1).any() else np.nan
    row["fpr@tau"] = float((s[y == 0] >= TAU).mean()) if (y == 0).any() else np.nan
    return row

# ── Hướng score: ĐĂNG KÝ TRƯỚC trong score() (= -PHD theo paper). KHÔNG lật theo nhãn val.
# Bản trước có block "AUROC(val) < 0.5 -> score = -score". Nó vi phạm
# statistical_branch_plan.md §4 và làm hỏng đúng thứ đang cần: nó BẢO ĐẢM AUROC(val) >= 0.5
# bằng cấu trúc, nên một model ra 0.31 sẽ được in thành 0.69. Binoculars/Fast-DetectGPT
# không được hưởng phép đó, nên leaderboard sẽ trộn hai protocol và PHD trông mạnh giả.
# AUROC(val) < 0.5 giờ chỉ là CẢNH BÁO để đi kiểm implementation.
FLIPPED = False   # giữ khoá này trong metrics.json để phân biệt với các lần chạy cũ

# Loại NaN (mẫu không ước lượng được PHD) TRƯỚC khi tính τ và mọi metric.
_n_nan = {}
for _name, _d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if _d is None:
        continue
    _bad = _d["score"].isna()
    _n_nan[_name] = int(_bad.sum())
    if _bad.any():
        print(f"[!] {_name}: loai {_bad.sum()}/{len(_d)} dong co score NaN "
              f"({_bad.sum() / len(_d):.2%}) - PHD khong uoc luong duoc")
        _d.drop(index=_d.index[_bad], inplace=True)
        _d.reset_index(drop=True, inplace=True)
if any(_n_nan.values()):
    print("    (bao con so nay trong paper: no la ti le mau bi loai, khong phai 0)")

_val_auc_raw = auroc(val_df["label"], val_df["score"]) if val_df is not None else np.nan

_std = float(np.std(val_df["score"])) if val_df is not None else 0.0
_auc = auroc(val_df["label"], val_df["score"]) if val_df is not None else float("nan")
print(f"std(score val) = {_std:.4f} | AUROC(val) = {_auc:.4f} | flipped={FLIPPED}")
if _std < 1e-6 or (not np.isnan(_auc) and _auc < 0.55):
    print("[!] CANH BAO: score gan nhu hang so / AUROC~0.5 - kiem tra lai model/scoring truoc khi tin bang so sanh.")

SMOKE_TEST = False   # cell merge chỉ nên chạy sau FULL; đổi True nếu bạn đang gộp kết quả SMOKE để debug
_, TAU = tpr_at_fpr(val_df["label"], val_df["score"], TARGET_FPR) if val_df is not None else (np.nan, np.nan)
print(f"tau (FPR={TARGET_FPR:.0%} tren val) = {TAU:.4f}")

rows = [r for r in [report_split("val", val_df), report_split("test_ood_model", ood_df),
                     report_split("test_fp", fp_df)] if r]
metrics_df = pd.DataFrame(rows).set_index("split")
pd.set_option("display.width", 200, "display.max_columns", 20)
print(metrics_df.round(4).to_string())

if not SMOKE_TEST:
    metrics_out = {"run": RUN_NAME, "model": MODEL_DESC, "tau": TAU, "target_fpr": TARGET_FPR,
                   "seed": 42, "flipped_sign": FLIPPED, "training_free": True, "dual_gpu": True,
                   "metrics": metrics_df.reset_index().to_dict("records")}
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2, default=float))
    for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
        if d is not None:
            d[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(
                OUT_DIR / f"pred_{nm}.parquet", index=False)
    print("da luu artifacts (gop 2 rank) ->", OUT_DIR)

In [ ]:
# 📊 Tóm tắt nhanh (đọc lại metrics.json vừa lưu)
import json as _json
from pathlib import Path as _Path
_p = _Path(f"artifacts_{RUN_NAME}/metrics.json")
if _p.exists():
    m = _json.loads(_p.read_text())
    print(f"RUN = {m['run']} | model = {m['model']} | tau = {m['tau']:.4f} | flipped_sign = {m['flipped_sign']}")
    for r in m["metrics"]:
        print(f"{r['split']:16} AUROC={r['auroc']:.4f}  TPR@FPR1%={r['tpr@fpr0.01']:.4f}  "
              f"TPR@FPR0.1%={r['tpr@fpr0.001']:.4f}  FPR@tau={r['fpr@tau']:.4f}")
else:
    print("Chưa có metrics.json — chạy cell FULL RUN trước (bỏ SMOKE=1).")


In [ ]:
# ── CHẨN ĐOÁN "CÓ SO SÁNH ĐƯỢC KHÔNG" — chạy sau cell metric ─────────────────
# Bảng metric ở trên không trả lời được ba câu mà bảng benchmark cuối cùng phải trả lời:
#
#   1. CHIỀU có nhất quán giữa các split? val và test_ood_model cùng phía 0.5 -> không có
#      bug đảo dấu. Nằm hai phía -> không có tín hiệu, và đừng đọc dấu của từng split.
#   2. AUROC có khác ĐOÁN MÒ không? SD của AUROC khi detector vô hiệu là
#      sqrt((n1+n2+1)/(12*n1*n2)) — với val 438+438 thì SD = 0.0195, nên 0.48 và 0.52
#      là CÙNG một kết luận: không phân biệt được.
#   3. TPR@FPR=1% so với mốc nào? Mốc đoán mò là ĐÚNG 0.01 (ROC của classifier vô hiệu là
#      đường chéo), KHÔNG phải 0. Nên 0.0337 là 3.4x đoán mò, không phải "gần như trượt".
#
# In cả mean score theo nhãn: nếu mean(AI) > mean(human) ở CẢ BA split thì hướng đăng ký
# trước là đúng và số thấp là bản chất, không phải lỗi implementation.
import glob
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

_CHANCE = {"tpr@fpr0.001": 0.001, "tpr@fpr0.01": 0.01, "tpr@fpr0.05": 0.05}


def _sd_null(n1, n2):
    return math.sqrt((n1 + n2 + 1) / (12 * n1 * n2))


def _p2(z):
    return math.erfc(abs(z) / math.sqrt(2))


_preds = sorted(p for p in glob.glob("**/pred_*.parquet", recursive=True)
                if "pred_matched" not in p)
if not _preds:
    print("khong tim thay pred_*.parquet -> chay cell metric truoc")
else:
    print("=" * 92)
    print("CHIEU SCORE THEO NHAN (mean; huong dang ky: cao = AI)")
    print("=" * 92)
    print(f"  {'split':18}{'n':>7}{'mean human':>13}{'mean AI':>12}{'gap':>11}  dung chieu?")
    _signs = {}
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        _sp = Path(_p).stem.replace("pred_", "")
        _h = _d.loc[_d["label"] == 0, "score"]
        _a = _d.loc[_d["label"] == 1, "score"]
        if len(_a) == 0:
            print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{'—':>12}{'—':>11}  (1 lop, bo qua)")
            continue
        _gap = _a.mean() - _h.mean()
        _signs[_sp] = _gap
        print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{_a.mean():12.4f}{_gap:+11.4f}"
              f"  {'CO' if _gap > 0 else 'KHONG'}")
    if len(_signs) >= 2:
        _ok = all(v > 0 for v in _signs.values()) or all(v < 0 for v in _signs.values())
        print(f"\n  -> chieu {'NHAT QUAN' if _ok else 'KHONG nhat quan'} qua "
              f"{len(_signs)} split"
              + ("" if _ok else "  <-- dau hieu KHONG CO TIN HIEU, dung doc dau tung split"))
        if _ok and all(v > 0 for v in _signs.values()):
            print("     dung huong dang ky truoc => implementation dung chieu; so thap la ban chat")

    print()
    print("=" * 92)
    print("CO HON DOAN MO KHONG")
    print("=" * 92)
    print(f"  {'split':18}{'AUROC':>9}{'SD_null':>10}{'z':>8}{'p':>9}  ket luan")
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        _sp = Path(_p).stem.replace("pred_", "")
        _n1 = int((_d["label"] == 1).sum())
        _n0 = int((_d["label"] == 0).sum())
        if _n1 == 0 or _n0 == 0:
            continue
        _a = float(roc_auc_score(_d["label"], _d["score"]))
        _sd = _sd_null(_n0, _n1)
        _z = (_a - 0.5) / _sd
        _v = "hon ngau nhien" if (_p2(_z) < 0.05 and _z > 0) else (
            "DUOI ngau nhien" if _p2(_z) < 0.05 else "KHONG phan biet duoc voi doan mo")
        print(f"  {_sp:18}{_a:9.4f}{_sd:10.4f}{_z:+8.2f}{_p2(_z):9.4f}  {_v}")

    print()
    print("=" * 92)
    print("TPR SO MOC DOAN MO (moc = dung bang FPR, khong phai 0)")
    print("=" * 92)
    _art = Path(_preds[0]).parent
    _mp = _art / "metrics.json"
    _m = json.loads(_mp.read_text()) if _mp.exists() else None
    if _m is None:
        print("  khong doc duoc metrics.json")
    else:
        print(f"  {'split':18}{'chi so':16}{'gia tri':>10}{'doan mo':>10}{'lan':>7}{'z':>8}")
        for _r in _m["metrics"]:
            _npos = _r.get("n_ai") or 0
            _nneg = (_r.get("n") or 0) - _npos
            if not _npos or not _nneg:
                continue
            for _k, _c in _CHANCE.items():
                _v = _r.get(_k)
                if _v is None or (isinstance(_v, float) and math.isnan(_v)):
                    continue
                _z = (_v - _c) / math.sqrt(_c * (1 - _c) / _npos)
                # nguong FPR duoc dat boi phan vi tren NEGATIVE -> so mau quyet dinh = _c*_nneg
                _ndec = _c * _nneg
                _flag = f"  <-- nguong do ~{_ndec:.1f} mau quyet dinh, DUNG BAO" if _ndec < 3 else ""
                print(f"  {_r['split']:18}{_k:16}{_v:10.4f}{_c:10.4f}{_v / _c:6.1f}x{_z:+8.1f}{_flag}")